# Create a SpatialData object and TissUUmaps export

All functions are located in [spatialdata_helper.py](spatialdata_helper.py) and [tissuumaps_helper.py](tissuumaps_helper.py) in the 
same folder as this notebook. To change the way data is loaded, please edit these
files.
The autoreload extension will update the python kernel automatically when these python 
files are changed.

## Load modules

In [15]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import os
os.add_dll_directory(r"C:\openslide\bin")  # Adjust path if needed
    
import spatialdata_helper
import tissuumaps_helper
import json

from spatialdata import SpatialData
from spatialdata.transformations import Affine, set_transformation, Identity, Sequence, Scale

from napari_spatialdata import Interactive

import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (12, 12)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Define transformations

We define transformation from DESI to H&E, as well as list 
of transformations applied to SCRINSHOT and DESI objects.

In [2]:
transformation_DESI_to_HnE = Affine(
    [
     [2.751790749353816, -0.0187122569641142, 53.8448401228362],
     [-0.0590478390637469, 2.72517753094655, 8.084978487269098],
     [0, 0, 1]
    ],
    input_axes=("x", "y"),
    output_axes=("x", "y"),
)

scale = Scale([250,350], axes=("x", "y")) #UPDATE BASED ON DESI COORDINATE SYSTEM
HnE_trans = {
    "DESI": Sequence([transformation_DESI_to_HnE.inverse(), scale]),
    "HnE": Identity(),
}

DESI_trans = {
    "DESI": Sequence([Identity(), scale]),
    "HnE": transformation_DESI_to_HnE,
}

## Create a spatial data object

### HnE data

Add image data (HnE)

In [3]:
# Main input folder
input_folder = "C:/Users/matsmi/OneDrive - Karolinska Institutet/Dokument/__STUDIES/006_MB_oxylipins_WP/data"
output_folder = "C:/Users/matsmi/OneDrive - Karolinska Institutet/Dokument/__STUDIES/006_MB_oxylipins_WP/results"

os.makedirs(output_folder, exist_ok=True)

# Path to your TIFF file
tif_path = fr"{input_folder}\HnE\01Nov_mouse_brain_middle_Octa.tif"

# Create the spatialdata object with the image
#sdata = SpatialData()

#HnE_image = spatialdata_helper.load_image(
#    sdata, 
#    tif_path, 
#    "HnE_image", 
#    transformations=DESI_trans
#)
#sdata

import numpy as np
import tifffile
from spatialdata.models import Image2DModel
sdata = SpatialData()

metadata = {
    "scale": {"x": 1, "y": 1},
    "translation": {"x": 0, "y": 0},
    "dims": ("c", "y", "x"),
}
HE_image = tifffile.imread(tif_path)
if HE_image.ndim == 3 and HE_image.shape[-1] in [3, 4]:  # RGB or RGBA images
    HE_image = np.transpose(HE_image, (2, 0, 1))

HE_image = Image2DModel.parse(
    data=HE_image,
    transformations=HnE_trans,
    **metadata
)

sdata.images["HnE_image"] = HE_image
sdata

SpatialData object
└── Images
      └── 'HnE_image': DataArray[cyx] (3, 883, 548)
with coordinate systems:
    ▸ 'DESI', with elements:
        HnE_image (Images)
    ▸ 'HnE', with elements:
        HnE_image (Images)

Add shapes (cell segmentation and anatomical labels)

In [ ]:
output_file = f"{output_folder}/HnE/anatomical_labels.geojson"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

input_folder = "C:/Users/matsmi/OneDrive - Karolinska Institutet/Dokument/__STUDIES/006_MB_oxylipins_WP/data"

spatialdata_helper.roi_to_geojson(
    sdata,
    fr"{input_folder}\HnE\anatomical_labels", 
    output_file=output_file,
    class_name="anatomical_labels",
    class_color=[0, 255, 0],
    transformations=HnE_trans # HnE_trans DESI_trans
)

sdata


SpatialData object
├── Images
│     └── 'HnE_image': DataArray[cyx] (3, 883, 548)
└── Shapes
      └── 'anatomical_labels': GeoDataFrame shape: (3, 1) (2D shapes)
with coordinate systems:
    ▸ 'DESI', with elements:
        HnE_image (Images), anatomical_labels (Shapes)
    ▸ 'HnE', with elements:
        HnE_image (Images), anatomical_labels (Shapes)

### DESI data

Add DESI data as Points and table of observations

In [17]:
input_folder = "C:/Users/matsmi/OneDrive - Karolinska Institutet/Dokument/__STUDIES/006_MB_oxylipins_WP/data"
input_path = fr"{input_folder}\DESI-MRM\images\01Nov_mouse_brain_middle_Octa\\"

spatialdata_helper.load_DESI (
    sdata, 
    input_path, 
    element_name="DESI",
    transformations=DESI_trans
)

sdata

10-KODA
12-HODA
12_13-DiHOME
12_13-Epoxy-9-KOME (EKODE)
13-HODE _or_ 12_13-EpOME _or_ 6(Z)_9(Z)-13-HODE
13-HOTrE
13-KODA
13-KODE _or_ 6(Z)_9(Z)-13-KODE
15_16-DiHODE
9-HODE _or_ 9_10-EpOME
9-KODE _or_ 10(E)_12(E)-9-KODE
9_10-DiHOME
9_10-Epoxy-13-HOME _or_ 12_13-Epoxy-9-HOME _or_ 9_10-Epoxy-11-HOME
9_12_13-TriHOME
Alpha Linolenic Acid
cis_trans-9_10-EpODA _or_ 9-HOME
CUDA [M-H]- _or_ CUDA [M-H]-
erythro_threo-9_10-DiHODA
FFA(18_1) - Oleic Acid
FFA(18_2)_[M-H]-
Hexose (gluc_fruct) [M+Cl]-
LPC(18_1)
LPC(18_2)
LPC(18_3)
LPI(18_1)
LPI(18_2)_[M-H]-
LPI(18_3)


SpatialData object
├── Images
│     ├── 'DESI_image': DataArray[cyx] (3, 334, 213)
│     └── 'HnE_image': DataArray[cyx] (3, 883, 548)
├── Shapes
│     ├── 'DESI': GeoDataFrame shape: (71142, 2) (2D shapes)
│     └── 'anatomical_labels': GeoDataFrame shape: (3, 1) (2D shapes)
└── Tables
      └── 'DESI_obs': AnnData (71142, 27)
with coordinate systems:
    ▸ 'DESI', with elements:
        DESI_image (Images), HnE_image (Images), DESI (Shapes), anatomical_labels (Shapes)
    ▸ 'HnE', with elements:
        DESI_image (Images), HnE_image (Images), DESI (Shapes), anatomical_labels (Shapes)

## Data visualization

Display the SpatialData content:

In [ ]:
sdata

Visualize in Napari:

In [ ]:
i = Interactive(sdata)
i.run()

Export to TissUUmaps:

In [ ]:
# Get all coordinate systems:
coordinate_systems = sdata.coordinate_systems

output_folder = f"{output_folder}/tissuumaps/01Nov_MB_Octa"
os.makedirs(output_folder, exist_ok=True)

for cs in coordinate_systems:
    print(f"Converting to TissUUmaps in coordinate system: {cs}")
    sdata_extended, tmap_file = tissuumaps_helper.spatialdata_to_tissuumaps(
        sdata, 
        elements = {
            "images": ["DESI_image", "HnE_image"],
            "shapes": ["anatomical_labels"],
            "tables": ["DESI_obs"],
            "points": [],
        },
        output_folder = output_folder,
        coordinate_system = cs
    )
    
    # Post-process the TissUUmaps file:
    with open(tmap_file, "r") as f:
        tmap_state = json.load(f)

    
    tmap_state["markerFiles"][0]["expectedHeader"]["scale_factor"] = 2

    for markerFile in tmap_state["markerFiles"]:
        for dropdownOption in markerFile["dropdownOptions"]:
            dropdownOption["name"] = dropdownOption["name"].replace("Gene expression: ", "")
    
    with open(tmap_file, "w") as f:
        json.dump(tmap_state, f, indent=2)

Converting to TissUUmaps in coordinate system: DESI
  - Adding image DESI_image
  - Adding image HnE_image


INFO:pyvips:VIPS: threadpool completed with 1 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 1 workers
INFO:pyvips:VIPS: vips__open_image_write: O_TMPFILE not available
INFO:pyvips:VIPS: vips__open_image_write: setting _O_TEMPORARY
INFO:pyvips:VIPS: vips__open_image_write: simple open
INFO:pyvips:VIPS: vips__open_image_write: O_TMPFILE not available
INFO:pyvips:VIPS: vips__open_image_write: setting _O_TEMPORARY
INFO:pyvips:VIPS: vips__open_image_write: simple open
INFO:pyvips:VIPS: vips__open_image_write: O_TMPFILE not available
INFO:pyvips:VIPS: vips__open_image_write: setting _O_TEMPORARY
INFO:pyvips:VIPS: vips__open_image_write: simple open
INFO:pyvips:VIPS: v

  - Adding shape anatomical_labels
  - Adding table DESI_obs
Converting to TissUUmaps in coordinate system: HnE


INFO:pyvips:VIPS: threadpool completed with 1 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 2 workers
INFO:pyvips:VIPS: threadpool completed with 14 workers
INFO:pyvips:VIPS: vips__open_image_write: O_TMPFILE not available
INFO:pyvips:VIPS: vips__open_image_write: setting _O_TEMPORARY
INFO:pyvips:VIPS: vips__open_image_write: simple open
INFO:pyvips:VIPS: vips__open_image_write: O_TMPFILE not available
INFO:pyvips:VIPS: vips__open_image_write: setting _O_TEMPORARY
INFO:pyvips:VIPS: vips__open_image_write: simple open
INFO:pyvips:VIPS: threadpool completed with 13 workers
INFO:pyvips:VIPS: threadpool completed with 13 workers
INFO:pyvips:VIPS: threadpool completed with 7 workers


  - Adding image DESI_image
  - Adding image HnE_image


INFO:pyvips:VIPS: threadpool completed with 13 workers
INFO:pyvips:VIPS: threadpool completed with 1 workers
INFO:pyvips:VIPS: threadpool completed with 13 workers
INFO:pyvips:VIPS: threadpool completed with 7 workers
INFO:pyvips:VIPS: threadpool completed with 3 workers


  - Adding shape anatomical_labels
  - Adding table DESI_obs


Save the SpatialData object to a zarr object on disk:

In [ ]:
# Save the updated spatialdata object:

output_path = f"{output_folder}/zarr/01Nov_MB_Octa.zarr"
print(output_path)
os.makedirs(os.path.dirname(output_path), exist_ok=True)
sdata.write(output_path, overwrite=True)

C:/Users/matsmi/OneDrive - Karolinska Institutet/Dokument/__STUDIES/006_MB_oxylipins_WP/results/output.zarr
INFO     The Zarr backing store has been changed from None the new file path: C:\Users\matsmi\OneDrive - Karolinska
         Institutet\Dokument\__STUDIES\006_MB_oxylipins_WP\results\output.zarr                                     
